# Práctica: Solr de Apache

**Materia:** Recuperación de la Información

**Apache Solr** ([https://solr.apache.org/](https://solr.apache.org/)) es un motor de búsqueda de texto completo de código abierto, construido sobre **Apache Lucene**. Este notebook explica su funcionamiento y muestra cómo se usa desde Python con la librería `pysolr`.

## ¿Para qué sirve?

Solr permite **indexar** grandes volúmenes de documentos (texto, JSON, XML, PDF, etc.) y hacer **búsquedas de texto completo** sobre ellos de forma muy rápida, con funciones avanzadas como facetas, resaltado de coincidencias (highlighting), sugerencias (autocompletar), búsqueda por relevancia y búsqueda geoespacial. Es la misma idea que vimos con Oracle Text, pero como un **servidor independiente** dedicado exclusivamente a búsqueda, en vez de una extensión de la base de datos.

## ¿Cómo funciona?

1. Solr corre como un **servidor** (un servicio Java independiente) que expone una **API REST/HTTP**.
2. Los documentos se envían al servidor para ser **indexados**: Solr los analiza con un *analyzer* (tokeniza el texto, quita palabras vacías, aplica stemming, etc., de forma similar a Oracle Text) y construye un **índice invertido** (palabra → lista de documentos que la contienen).
3. Las búsquedas también se hacen por HTTP: el cliente (por ejemplo, un script en Python usando `pysolr`) envía una consulta al servidor, y Solr responde con los documentos que coinciden, ordenados por relevancia (usando el algoritmo BM25 por defecto).
4. Solr organiza los datos en **colecciones** (equivalentes a una tabla o índice), cada una con su propio **esquema** que define los campos y cómo se analiza cada uno.

## ¿Qué tipo de consultas se pueden realizar?

- **Búsqueda de palabra o frase**: `q=texto:"aprendizaje automatico"`.
- **Búsqueda por campo específico**: `q=titulo:python`.
- **Operadores booleanos**: `q=python AND (machine OR deep) NOT java`.
- **Comodines y fuzzy**: `q=pyth*`, `q=pythom~1` (tolera un error de escritura).
- **Rangos**: `q=fecha:[2020-01-01 TO 2023-12-31]` o `precio:[10 TO 50]`.
- **Facetas**: contar cuántos documentos hay por categoría (`facet.field=categoria`), útil para filtros tipo "e-commerce".
- **Resaltado (highlighting)**: devolver el fragmento de texto donde aparece la coincidencia, marcado.
- **Ordenar por relevancia o por campo** (`sort=fecha desc`).

## ⚠️ Por qué este notebook no muestra resultados "reales" (y por qué eso es normal)

A diferencia de Oracle Text (que es parte de la base de datos y ya viene instalado) o de OpenCV/librosa (que son solo librerías de Python), **Apache Solr es un servidor aparte**: requiere tener **Java 11+** instalado y **arrancar un proceso de Solr** que quede escuchando en un puerto (por defecto `8983`) *antes* de poder consultarlo desde Python. Si ese servidor no está corriendo —como en este entorno de preparación, y probablemente en tu computadora si nunca lo has instalado— **es imposible indexar o buscar nada de verdad**, sin importar qué tan bien esté escrito el código Python. Esto **no es un error de la práctica ni de `pysolr`**: es el comportamiento esperado y correcto (`solr.ping()` falla porque no hay nadie escuchando en ese puerto).

El código de abajo con `pysolr` es **correcto y funcional tal cual está**. Para que sí muestre resultados reales en tu computadora, instala y arranca Solr primero:

```bash
# 1. Descargar Apache Solr (requiere Java 11+ instalado) desde https://solr.apache.org/downloads.html
# 2. Arrancar un servidor de ejemplo:
bin/solr start -e cloud
# (en Windows: bin\solr.cmd start -e cloud)

# 3. Instalar el cliente de Python
pip install pysolr
```

El notebook **detecta automáticamente** si el servidor está disponible (`solr.ping()`); si no lo está, imprime el motivo exacto y muestra el código de referencia equivalente sin fallar ni quedarse trabado.

In [1]:
import pysolr

SOLR_URL = 'http://localhost:8983/solr/practica_ri'
solr = pysolr.Solr(SOLR_URL, timeout=3)

SOLR_DISPONIBLE = False
try:
    solr.ping()
    SOLR_DISPONIBLE = True
    print('Servidor Solr disponible en', SOLR_URL)
except Exception as e:
    print(f'Servidor Solr no disponible en este entorno: {e}')
    print('Esto es esperado (ver la explicacion en la celda de arriba): no hay un proceso de Solr corriendo en el puerto 8983.')
    print('Se mostrara el codigo de referencia sin ejecutar operaciones reales contra el servidor.')

Servidor Solr no disponible en este entorno: Failed to connect to server at http://localhost:8983/solr/practica_ri/admin/ping/?: HTTPConnectionPool(host='localhost', port=8983): Max retries exceeded with url: /solr/practica_ri/admin/ping/ (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8983): Failed to establish a new connection: [Errno 111] Connection refused"))
Esto es esperado (ver la explicacion en la celda de arriba): no hay un proceso de Solr corriendo en el puerto 8983.
Se mostrara el codigo de referencia sin ejecutar operaciones reales contra el servidor.


### Indexar documentos de ejemplo

In [2]:
documentos = [
    {'id': '1', 'titulo': 'Introduccion a Oracle Text', 'contenido': 'Oracle Text permite busquedas de texto completo dentro de SQL.'},
    {'id': '2', 'titulo': 'Fundamentos de Apache Solr', 'contenido': 'Solr es un motor de busqueda basado en Apache Lucene.'},
    {'id': '3', 'titulo': 'Procesamiento de audio con librosa', 'contenido': 'librosa permite analizar espectrogramas y extraer caracteristicas de audio.'},
]

if SOLR_DISPONIBLE:
    solr.add(documentos)
    solr.commit()
    print(f'{len(documentos)} documentos indexados correctamente')
else:
    print('Omitido (requiere servidor Solr). Codigo equivalente:')
    print('  solr.add(documentos)')
    print('  solr.commit()')

Omitido (requiere servidor Solr). Codigo equivalente:
  solr.add(documentos)
  solr.commit()


### Búsqueda simple

In [3]:
if SOLR_DISPONIBLE:
    resultados = solr.search('contenido:busqueda')
    print(f'{len(resultados)} resultados encontrados')
    for doc in resultados:
        print(' -', doc['titulo'])
else:
    print('Omitido (requiere servidor Solr). Codigo equivalente:')
    print("  resultados = solr.search('contenido:busqueda')")
    print('  for doc in resultados: print(doc["titulo"])')

Omitido (requiere servidor Solr). Codigo equivalente:
  resultados = solr.search('contenido:busqueda')
  for doc in resultados: print(doc["titulo"])


### Búsqueda con facetas

Las facetas permiten, por ejemplo, contar cuántos documentos hay agrupados por un campo (como categorías en una tienda en línea).

In [4]:
if SOLR_DISPONIBLE:
    resultados = solr.search('*:*', **{'facet': 'true', 'facet.field': 'titulo'})
    print(resultados.facets)
else:
    print('Omitido (requiere servidor Solr). Codigo equivalente:')
    print("  solr.search('*:*', facet='true', **{'facet.field': 'titulo'})")

Omitido (requiere servidor Solr). Codigo equivalente:
  solr.search('*:*', facet='true', **{'facet.field': 'titulo'})


### Resaltado de coincidencias (highlighting)

In [5]:
if SOLR_DISPONIBLE:
    resultados = solr.search('contenido:Solr', **{'hl': 'true', 'hl.fl': 'contenido'})
    print(resultados.highlighting)
else:
    print('Omitido (requiere servidor Solr). Codigo equivalente:')
    print("  solr.search('contenido:Solr', hl='true', **{'hl.fl': 'contenido'})")

Omitido (requiere servidor Solr). Codigo equivalente:
  solr.search('contenido:Solr', hl='true', **{'hl.fl': 'contenido'})


## Conclusión

El que este notebook no muestre resultados en vivo no significa que la práctica esté incompleta: significa que Solr, al ser un servidor independiente, necesita ese servidor arrancado para poder usarse, algo que Oracle Text no necesita (por venir integrado en la base de datos). El flujo mostrado (`add`, `commit`, `search`, facetas, highlighting) es el código real que se usa para integrar Solr en una aplicación Python, y es directamente comparable con lo visto en Oracle Text: ambos indexan texto y permiten consultarlo con operadores booleanos, frases, comodines y puntajes de relevancia, aunque Solr lo hace como un servicio de búsqueda independiente en vez de una extensión de la base de datos.